In [1]:
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.metrics import confusion_matrix
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from utils.data_loader import load_volvo
from scipy.stats import chi2
from scipy.stats import binomtest
from scipy import stats

In [2]:
# -----------------------
# TranAD
# -----------------------
TranAD_scores = pd.read_csv("../Results/Volvo TranAD/scores.csv")
TranAD_CV = pd.read_csv("../Results/Volvo TranAD/results_cross_validation.csv")

TranAD_threshold = TranAD_CV["mean_threshold"].iloc[0]
TranAD_scores = TranAD_scores.to_numpy().ravel()


# -----------------------
# DeepSVDD
# -----------------------
DeepSVDD_scores = pd.read_csv("../Results/Volvo DeepSVDD/scores.csv")
DeepSVDD_CV = pd.read_csv("../Results/Volvo DeepSVDD/results_cross_validation.csv")

DeepSVDD_threshold = DeepSVDD_CV["mean_threshold"].iloc[0]
DeepSVDD_scores = DeepSVDD_scores.to_numpy().ravel()



# -----------------------
# COPOD
# -----------------------
COPOD_scores = pd.read_csv("../Results/Volvo COPOD/scores.csv")
COPOD_CV = pd.read_csv("../Results/Volvo COPOD/results_cross_validation.csv")

COPOD_threshold = COPOD_CV["mean_threshold"].iloc[0]
COPOD_scores = COPOD_scores.to_numpy().ravel()


# -----------------------
# COUTA
# -----------------------
COUTA_scores = pd.read_csv("../Results/Volvo COUTA/scores.csv")
COUTA_CV = pd.read_csv("../Results/Volvo COUTA/results_cross_validation.csv")

COUTA_threshold = COUTA_CV["mean_threshold"].iloc[0]
COUTA_scores = COUTA_scores.to_numpy().ravel()


# -----------------------
# LOF
# -----------------------
LOF_scores = pd.read_csv("../Results/Volvo LOF/scores.csv")
LOF_CV = pd.read_csv("../Results/Volvo LOF/results_cross_validation.csv")

LOF_threshold = LOF_CV["mean_threshold"].iloc[0]
LOF_scores = LOF_scores.to_numpy().ravel()


# -----------------------
# TcnED
# -----------------------
TcnED_scores = pd.read_csv("../Results/Volvo TcnED/scores.csv")
TcnED_CV = pd.read_csv("../Results/Volvo TcnED/results_cross_validation.csv")

TcnED_threshold = TcnED_CV["mean_threshold"].iloc[0]
TcnED_scores = TcnED_scores.to_numpy().ravel()

In [3]:
# Get predictions from scores and threshold
TranAD_predictions = (TranAD_scores>TranAD_threshold).astype(int)
DeepSVDD_predictions = (DeepSVDD_scores > DeepSVDD_threshold).astype(int)
COPOD_predictions = (COPOD_scores > COPOD_threshold).astype(int)
COUTA_predictions = (COUTA_scores > COUTA_threshold).astype(int)
LOF_predictions = (LOF_scores > LOF_threshold).astype(int)
TcnED_predictions = (TcnED_scores > TcnED_threshold).astype(int)

In [4]:
x_train, y_train, x_test, y_test, timestamps_train, timestamps_test, df_train, df_test = load_volvo(
    "../Dataset/Processed Dataset/volvoTrain.csv",
    "../Dataset/Processed Dataset/volvoTest.csv"
)

In [5]:
TranAD_correct = TranAD_predictions == y_test
DeepSVDD_correct = DeepSVDD_predictions == y_test
COPOD_correct = COPOD_predictions == y_test
COUTA_correct = COUTA_predictions == y_test
LOF_correct = LOF_predictions == y_test
TcnED_correct = TcnED_predictions == y_test

In [6]:
table = confusion_matrix(TranAD_correct, LOF_correct)
result = mcnemar(table, exact=True)

print(result.pvalue)

5.739718509874451e-42


In [7]:
predictions_dict = {
    "TranAD": TranAD_correct,
    "DeepSVDD": DeepSVDD_correct,
    "COPOD": COPOD_correct,
    "COUTA": COUTA_correct,
    "LOF": LOF_correct,
    "TcnED": TcnED_correct,
}

results = []

for (model1_name, model1_predictions), (model2_name, model2_predictions) in combinations(predictions_dict.items(), 2):

    # Building contingency table
    model1_1_model2_0 = 0
    model1_1_model2_1 = 0
    model1_0_model2_0 = 0
    model1_0_model2_1 = 0

    for i in range(len(model1_predictions)):
        if model1_predictions[i] == 1 and model2_predictions[i] == 1:
            model1_1_model2_1 += 1
        elif model1_predictions[i] == 1 and model2_predictions[i] == 0:
            model1_1_model2_0 += 1
        elif model1_predictions[i] == 0 and model2_predictions[i] == 1:
            model1_0_model2_1 += 1
        elif model1_predictions[i] == 0 and model2_predictions[i] == 0:
            model1_0_model2_0 += 1
        else:
            print("Unhandled case")
    a = model1_1_model2_1
    b = model1_1_model2_0
    c = model1_0_model2_1
    d = model1_0_model2_0

    # mcnemar() uses this for doing McNemar formula: n1, n2 = table[0, 1], table[1, 0] (so b and c)
    # !! not same order as McNemar OG paper !!
    # If a and d are replaced with 0, scores don't change so should be correct
    table = [[a, b],
             [c, d]]

    print(table)

    # manual McNemar (verification)
    chiSquared = ((abs(b-c)-1)**2)/(b+c)
    chi_p_value = stats.chi2.sf(chiSquared, df=1)


    xSquared = ((abs(b-c)-1)**2)/(b+c)
    manual_p_value = 1 - chi2.cdf(xSquared, df=1)
    result = mcnemar(table, exact=False)

    print("b+c = ", b+c)
    print(f"manual p-value: {manual_p_value:.3e}")

    results.append({
        "Model 1": model1_name,
        "Model 2": model2_name,
        "p-value": result.pvalue,

    })
    print(f"{model1_name} vs {model2_name}")
    print(f"p-value: {result.pvalue:.3e}")
    print(f"xSquared: {result.statistic:.3e}")
    print(f"xSquared: {xSquared:.3e}")
    print(f"chi_p: {chi_p_value:.3e}")
    print("-" * 40)

results_df = pd.DataFrame(results)
print(results_df)

results_df.to_csv("McNemar_test_volvo.csv", index=False)

[[1381190, 3], [1, 161]]
b+c =  4
manual p-value: 6.171e-01
TranAD vs DeepSVDD
p-value: 6.171e-01
xSquared: 2.500e-01
xSquared: 2.500e-01
chi_p: 6.171e-01
----------------------------------------
[[1369091, 12102], [45, 117]]
b+c =  12147
manual p-value: 0.000e+00
TranAD vs COPOD
p-value: 0.000e+00
xSquared: 1.197e+04
xSquared: 1.197e+04
chi_p: 0.000e+00
----------------------------------------
[[1381189, 4], [4, 158]]
b+c =  8
manual p-value: 7.237e-01
TranAD vs COUTA
p-value: 7.237e-01
xSquared: 1.250e-01
xSquared: 1.250e-01
chi_p: 7.237e-01
----------------------------------------
[[1381193, 0], [138, 24]]
b+c =  138
manual p-value: 0.000e+00
TranAD vs LOF
p-value: 1.988e-31
xSquared: 1.360e+02
xSquared: 1.360e+02
chi_p: 1.988e-31
----------------------------------------
[[1381188, 5], [0, 162]]
b+c =  5
manual p-value: 7.364e-02
TranAD vs TcnED
p-value: 7.364e-02
xSquared: 3.200e+00
xSquared: 3.200e+00
chi_p: 7.364e-02
----------------------------------------
[[1369091, 12100], [45

In [8]:
alpha = 0.05

models = list(predictions_dict.keys())

# init matrix with blanks on diagonal
sig_matrix = {m: {n: "\\Blank" for n in models} for m in models}

# fill from results_df
for _, row in results_df.iterrows():
    m1 = row["Model 1"]
    m2 = row["Model 2"]
    p = row["p-value"]

    sig = "\\Sig" if p < alpha else "\\NS"

    sig_matrix[m1][m2] = sig
    sig_matrix[m2][m1] = sig  # symmetric

# print LaTeX table
print("\\textbf{" + "} & \\textbf{".join(models) + "}\\\\")
print("\\hline")

for m1 in models:
    row = [f"\\textbf{{{m1}}}"]
    for m2 in models:
        row.append(sig_matrix[m1][m2])
    print(" & ".join(row) + " \\\\")

\textbf{TranAD} & \textbf{DeepSVDD} & \textbf{COPOD} & \textbf{COUTA} & \textbf{LOF} & \textbf{TcnED}\\
\hline
\textbf{TranAD} & \Blank & \NS & \Sig & \NS & \Sig & \NS \\
\textbf{DeepSVDD} & \NS & \Blank & \Sig & \NS & \Sig & \NS \\
\textbf{COPOD} & \Sig & \Sig & \Blank & \Sig & \Sig & \Sig \\
\textbf{COUTA} & \NS & \NS & \Sig & \Blank & \Sig & \NS \\
\textbf{LOF} & \Sig & \Sig & \Sig & \Sig & \Blank & \Sig \\
\textbf{TcnED} & \NS & \NS & \Sig & \NS & \Sig & \Blank \\
